# Reason Tools

> Exposes `reason_over` as a single OpenAIâ€‘style function descriptor.

In [ ]:
#| default_exp tools.reason

In [ ]:
#| export
import json
from typing import Dict, Tuple
from rdflib import Graph, ConjunctiveGraph
from cogitarelink.core.debug import get_logger
from cogitarelink.reason.sandbox import reason_over
from cogitarelink.core.graph import GraphManager
from cogitarelink.reason.prov import wrap_patch_with_prov
from pyshacl import validate

In [ ]:
#| export
log=get_logger("tool.reason")
gm = GraphManager(use_rdflib=True)   # shared, memoryâ€‘safe

In [ ]:
#| export
FUNCTION_SPEC:Dict = {
  "name":"reason_over",
  "description":"Run SHACL rules or SPARQL CONSTRUCT on JSONâ€‘LD and update memory",
  "parameters":{
    "type":"object",
    "properties":{
      "jsonld":{"type":"string","description":"Primary JSON-LD input"},
      "shapes_turtle":{"type":"string","description":"Optional SHACL shapes in turtle"},
      "query":{"type":"string","description":"Optional SPARQL CONSTRUCT query"}
    },
    "required":["jsonld"]
  }
}

In [ ]:
#| export
def reason_tool(jsonld:str,
                shapes_turtle:str|None=None,
                query:str|None=None) -> str:
    patch, summary = reason_over(jsonld=jsonld,
                                 shapes_turtle=shapes_turtle,
                                 query=query)
    # Convert JSON-LD to nquads format for ingestion
    from rdflib import Graph, ConjunctiveGraph
    temp_g = ConjunctiveGraph()  # Use ConjunctiveGraph for nquads
    temp_g.parse(data=patch, format="json-ld")
    nquads = temp_g.serialize(format="nquads")
    gm.ingest_nquads(nquads, graph_id="patch")
    return summary

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()